# vLLM Bridge — Integration Test

End-to-end validation of `boot_vllm` on a real GPU. Runs the v4 Driver protocol's vLLM backend against `meta-llama/Llama-3.2-1B`, compares per-hook activations and next-token argmax against the HF transformers backend, and exercises the affine intervention path.

**Branch:** `dev-4.x`. **Hardware:** any CUDA GPU with ≥10 GB VRAM (free Colab T4 is sufficient).

## Scope: observation + mutation

This source extends past vllm-lens's observation-only scope. Each capture hook applies an affine transform (`output = output * scale + bias`, default identity) and returns the modified tensor, so interventions propagate to downstream layers. Step 6 below is the load-bearing verification that this works end-to-end under `torch.compile` + CUDA graphs — unit tests can't reach the compiled-graph path.

## What this validates
1. `boot_vllm` returns a `RemoteBridge` end-to-end.
2. `run_with_cache` populates an `ActivationCache` from the vLLM worker via `collective_rpc`.
3. Greedy next-token argmax matches `boot_transformers` (greedy parity).
4. Per-fireable-hook relative L2 < 5e-3 vs HF (hook-fidelity gate; failures abort).
5. **Mutation smoke** (load-bearing): suppressing `embed.hook_out` zeros the cache there, shifts the argmax, and reverts cleanly on the next forward.
6. GPU release after `del bridge`.

## Setup

1. **Runtime → Change runtime type → GPU** (T4 / L4 / A100 all work).
2. **Secrets → add `HF_TOKEN`** with a token that has access to `meta-llama/Llama-3.2-1B` (gated).

The environment cell below patches `sys.stdout.fileno` because ipykernel's captured stdout doesn't expose a real file descriptor and vLLM's worker init calls `fileno()`. Without the patch, Step 2 fails with `UnsupportedOperation: fileno`.

In [ ]:
# Install vllm and TransformerLens @ feature/driver-system. ~3-5 minutes.
# vllm pinned to 0.20.2 — the version the internal-API walks in
# transformer_lens/model_bridge/sources/vllm/{plugin,internals}.py were validated against.
# vLLM rearranges its internal class paths every 4-6 weeks; re-validate before bumping.
%pip install -q "vllm==0.20.2"
%pip install -q git+https://github.com/TransformerLensOrg/TransformerLens.git@feature/driver-system

In [ ]:
import gc
import os
import sys

import torch

# HF_TOKEN comes from Colab secrets. Falls back to env var for non-Colab runs.
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except (ImportError, Exception):
    pass
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets (gear icon, left sidebar)."

# Colab/Jupyter compatibility: ipykernel's stdout doesn't expose a fileno();
# vLLM's worker init calls sys.stdout.fileno() during parallel-state setup
# and crashes with UnsupportedOperation: fileno. Patch fileno to return the
# underlying process FDs (1, 2) — Colab writes back to those anyway.
if "ipykernel" in sys.modules:
    sys.stdout.fileno = lambda: 1  # type: ignore[method-assign]
    sys.stderr.fileno = lambda: 2  # type: ignore[method-assign]

# Read the installed vllm version from package metadata, NOT `import vllm` —
# importing vllm loads its CUDA C extension (vllm._C), which is exactly what
# fails with `libcudart.so.NN not found` when a newer wheel built for a CUDA
# version Colab doesn't ship gets installed. Metadata read works regardless.
from importlib.metadata import PackageNotFoundError
from importlib.metadata import version as _pkg_version

_PINNED_VLLM = "0.20.2"
try:
    _vllm_ver = _pkg_version("vllm")
    print(f"vllm version: {_vllm_ver}")
    if _vllm_ver != _PINNED_VLLM:
        print(
            f"⚠ expected vllm=={_PINNED_VLLM} (the version the capture plugin is validated "
            f"against); got {_vllm_ver}. Newer wheels target a CUDA the Colab image may not "
            "ship (→ libcudart.so error at boot) and may move vLLM internals the plugin walks. "
            f"Re-pin with %pip install -q 'vllm=={_PINNED_VLLM}' and restart the runtime."
        )
except PackageNotFoundError:
    print("⚠ vllm is not installed — run the install cell above.")

MODEL = "meta-llama/Llama-3.2-1B"
PROMPT = "The quick brown fox jumps over the"
DTYPE = torch.float16
torch.manual_seed(0)

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — abort'}")
assert torch.cuda.is_available(), "GPU runtime required."

## Step 1 — HF reference

Boot the transformers backend first, capture activations and argmax, then drop it so vLLM has the GPU to itself.

In [ ]:
from transformers import AutoConfig

from transformer_lens.model_bridge.sources.transformers import boot as boot_transformers
from transformer_lens.model_bridge.sources.vllm.overlays import get_overlay

# Compute the expected vLLM-fireable hook set from the overlay so HF captures
# only what we'll compare. Avoids hundreds of MB of CPU clones for hooks vLLM
# doesn't expose.
_hf_preview = AutoConfig.from_pretrained(MODEL, token=os.environ["HF_TOKEN"])
_overlay = get_overlay(_hf_preview.architectures[0])
EXPECTED_HOOKS = set(_overlay.capture_specs(_hf_preview).keys())
print(f"Expecting {len(EXPECTED_HOOKS)} fireable hook(s) per vLLM overlay.")

bridge_hf = boot_transformers(MODEL, dtype=DTYPE).to("cuda")
tokens = bridge_hf.to_tokens(PROMPT)
# no_grad drops the autograd graph; without this, the forward-pass intermediates
# stay alive (~8+ GiB for a 1B model) and starve vLLM's KV cache allocation later.
with torch.no_grad():
    logits_hf, cache_hf = bridge_hf.run_with_cache(
        tokens, names_filter=lambda name: name in EXPECTED_HOOKS
    )
argmax_hf = int(logits_hf[0, -1].argmax().item())

cache_hf_cpu = {name: t.detach().cpu().clone() for name, t in cache_hf.cache_dict.items()}
next_token_hf = bridge_hf.tokenizer.decode([argmax_hf])
print(f"HF argmax token id: {argmax_hf} → {next_token_hf!r}")
print(f"HF cache: {len(cache_hf_cpu)} entries (filtered to overlay's fireable set)")

# Move parameters to CPU before deletion to force release even if a reference
# cycle persists. del + gc.collect alone has been observed to leak ~6 GiB here.
bridge_hf.to("cpu")
del bridge_hf, logits_hf, cache_hf, tokens
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory after HF release: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Step 2 — Boot vLLM bridge

`boot_vllm` constructs the LLM, monkey-patches `Worker.load_model` pre-compile to install capture hooks, then wraps it in a `RemoteBridge`.

In [ ]:
from transformer_lens.model_bridge.remote_bridge import RemoteBridge

# max_model_len=2048 caps the KV cache reservation. Llama-3.2-1B's native
# context is 131072 (128k) — the default reservation is ~4 GiB and overshoots
# the free T4 budget. The test prompt is ~10 tokens, so 2048 is plenty.
bridge = RemoteBridge.boot_vllm(
    MODEL,
    dtype=DTYPE,
    gpu_memory_utilization=0.5,
    max_model_len=2048,
)
assert isinstance(bridge, RemoteBridge), f"Expected RemoteBridge, got {type(bridge).__name__}"
print(f"Architecture: {bridge.cfg.architecture}")
print(f"Fireable hooks: {len(bridge._driver.supported_hook_points)}")
print(f"Non-fireable hooks (fused kernels): {len(bridge._driver.non_fireable_hook_points)}")

## Step 3 — Capture pipeline

Run a single forward, populate the cache via `collective_rpc → tl_read_captures`, and sanity-check shapes.

In [ ]:
tokens = bridge.to_tokens(PROMPT)
logits_vllm, cache_vllm = bridge.run_with_cache(tokens)
argmax_vllm = int(logits_vllm[0, -1].argmax().item())
next_token_vllm = bridge.tokenizer.decode([argmax_vllm])

print(f"vLLM argmax token id: {argmax_vllm} → {next_token_vllm!r}")
print(f"vLLM cache entries: {len(cache_vllm.cache_dict)}")

for name in sorted(bridge._driver.supported_hook_points)[:5]:
    t = cache_vllm[name]
    finite_pct = 100 * torch.isfinite(t).float().mean().item()
    print(f"  {name}: shape={tuple(t.shape)}, dtype={t.dtype}, finite={finite_pct:.1f}%")

## Step 4 — Greedy parity 

vLLM and HF must produce the same next-token argmax on the same prompt.

In [ ]:
parity = argmax_vllm == argmax_hf
status = "✅ PASS" if parity else "❌ FAIL"
print(f"{status}: HF→{argmax_hf} ({next_token_hf!r}) vs vLLM→{argmax_vllm} ({next_token_vllm!r})")
assert parity, "Greedy parity failed — kernel divergence or overlay misconfiguration."

## Step 5 — Per-hook L2 (acceptance gate)

Target is relative L2 < 5e-3 in fp16 for every fireable hook. One hook remains exempted from the strict gate:

- **`ln_final.hook_normalized`** — vLLM's `model.norm` is invoked as part of the fused-residual norm kernel and the captured value scales ~2× HF's. Open investigation (likely a residual-fusion semantic discrepancy at the model boundary rather than a real divergence; argmax + downstream parity work correctly).

All other fireable hooks (including `blocks.{i}.hook_out`, which is now materialized from vLLM's `(mlp_delta, residual)` tuple) must be within target.

In [ ]:
TARGET_REL_L2 = 5e-3
# Hooks where vLLM's architecture diverges from HF at the capture point in
# ways we haven't yet reconciled. Only ln_final remains after the layer-hook
# materialization landed.
_RESIDUAL_FUSION_DIVERGENT = {"ln_final.hook_normalized"}

rows = []
for name in sorted(bridge._driver.supported_hook_points):
    if name not in cache_hf_cpu:
        rows.append((name, None, None, "missing in HF cache"))
        continue
    t_vllm = cache_vllm[name].detach().cpu().float()
    t_hf = cache_hf_cpu[name].float()
    if t_vllm.shape != t_hf.shape:
        rows.append((name, None, None, f"shape {tuple(t_vllm.shape)} vs HF {tuple(t_hf.shape)}"))
        continue
    diff = (t_vllm - t_hf).norm().item()
    base = t_hf.norm().item() or 1.0
    rel = diff / base
    if rel < TARGET_REL_L2:
        mark = "✅"
        note = ""
    elif name in _RESIDUAL_FUSION_DIVERGENT:
        mark = "⚠"
        note = "residual-fusion (expected)"
    else:
        mark = "❌"
        note = ""
    rows.append((name, rel, mark, note))

print(f"{'hook':<40} {'rel L2':>10}  status  note")
print("-" * 80)
for name, rel, mark, note in rows:
    rel_str = f"{rel:.3e}" if isinstance(rel, float) else "—"
    print(f"{name:<40} {rel_str:>10}  {mark or '—':<6}  {note}")

# Strict gate: only the non-residual-fusion hooks need to be within target.
failed = [(name, rel) for name, rel, _, _ in rows
          if rel is not None and rel >= TARGET_REL_L2 and name not in _RESIDUAL_FUSION_DIVERGENT]
assert not failed, f"Hooks exceeded fp16 drift (target {TARGET_REL_L2}): {failed}"
missing = [(name, note) for name, rel, _, note in rows if rel is None]
assert not missing, f"Hooks missing or shape-mismatched vs HF: {missing}"

## Step 6 — Intervention smoke (load-bearing)

**This cell is the load-bearing verification for the vLLM source's mutation claim.** Unit tests only exercise the dispatch protocol (mocked LLM); they cannot prove that the affine math (`output = output * scale + bias`) traces correctly through `torch.compile` and propagates to downstream layers under CUDA-graph replay. That guarantee depends on this cell passing.

The test: zero out `embed.hook_out` via `{"op": "suppress"}`. If the mutation path works, (a) the cache shows zeros there, (b) the next-token argmax shifts vs the clean run, and (c) the immediately-following clean forward reverts (no sticky state).

In [ ]:
# (a) suppress: zero the embedding output. Multiplicative op (scale=0).
logits_suppressed, cache_suppressed = bridge.run_with_cache(
    tokens,
    intervene={"embed.hook_out": {"op": "suppress"}},
)
argmax_suppressed = int(logits_suppressed[0, -1].argmax().item())
embed_norm = cache_suppressed["embed.hook_out"].abs().max().item()
print(f"suppress: embed |max|={embed_norm:.6f} (→0)  argmax {argmax_vllm} → {argmax_suppressed}")
assert embed_norm == 0.0, "suppress did not zero embed.hook_out"
assert argmax_suppressed != argmax_vllm, "suppress did not change the prediction"

# (b) set: force embed to a constant vector. Additive op (scale=0, bias=value).
d_model = bridge.cfg.d_model
set_value = [0.5] * d_model
logits_set, cache_set = bridge.run_with_cache(
    tokens,
    intervene={"embed.hook_out": {"op": "set", "value": set_value}},
)
set_max = cache_set["embed.hook_out"].abs().max().item()
print(f"set:      embed |max|={set_max:.6f} (→0.5)  argmax {argmax_vllm} → {int(logits_set[0, -1].argmax().item())}")
assert abs(set_max - 0.5) < 1e-2, "set did not write the constant value to embed.hook_out"

# (c) reset: a clean forward must revert — interventions are not sticky.
logits_revert, _ = bridge.run_with_cache(tokens)
argmax_revert = int(logits_revert[0, -1].argmax().item())
print(f"reset:    argmax → {argmax_revert}  matches clean: {argmax_revert == argmax_vllm}")
assert argmax_revert == argmax_vllm, "intervention persisted across calls — reset path broken"

## Step 7 — Hook fires exactly once

A correctness gate for the capture mechanism: under `torch.compile` + CUDA graphs, each capture hook must fire exactly once per forward. A double-fire would silently overwrite the buffer (and with multi-token decode, write the wrong step's activation). The plugin increments a shared GPU counter on every hook fire; after one forward it should equal the number of installed hooks.

In [ ]:
bridge._driver._llm.collective_rpc("tl_reset_counter")
bridge.run_with_cache(tokens)
fire_count = bridge._driver._llm.collective_rpc("tl_read_counter")[0]
expected = len(bridge._driver.supported_hook_points)
print(f"Hook fires: {fire_count}  expected (one per hook): {expected}")
assert fire_count == expected, (
    f"Expected {expected} fires (one per installed hook), got {fire_count} — "
    "a hook fired more than once under compile, or some didn't fire."
)
print("✅ every capture hook fired exactly once.")

## Step 8 — Stream safety

Capturing the same prompt repeatedly must be deterministic — back-to-back forwards through the compiled graph should produce bitwise-identical captures. Drift here would indicate a stale-buffer race or non-deterministic kernel path in the capture machinery.

In [ ]:
refs = []
for _ in range(5):
    _, cache_i = bridge.run_with_cache(tokens)
    refs.append(cache_i["embed.hook_out"].detach().cpu().clone())
for i, c in enumerate(refs[1:], start=1):
    assert torch.equal(refs[0], c), f"Run {i} differs from run 0 — non-deterministic capture"
print("✅ 5 back-to-back captures are bitwise identical.")

## Step 9 — Lifetime

`bridge.close()` is responsible for releasing **our** resources:
- detaches the per-Worker capture hooks via `tl_remove_hooks`
- tears down vLLM's distributed environment (`destroy_distributed_environment`)

It cannot release **vLLM's** internal state in 0.20.2 — there's no `LLM.shutdown()` API. Model weights, KV cache pool, and Inductor compile cache stay resident in PyTorch's caching allocator until process exit. A clean Colab `Runtime → Restart session` is the only way to fully reclaim GPU memory.

What we *can* verify: `bridge.close()` ran without error and didn't leak more memory. The residual is informational, not a gate.

In [ ]:
before = torch.cuda.memory_allocated() / 1e9
bridge.close()  # detaches hooks + tears down vLLM distributed state
del bridge, logits_vllm, cache_vllm, logits_suppressed, cache_suppressed, logits_revert
gc.collect(); torch.cuda.empty_cache()
after = torch.cuda.memory_allocated() / 1e9
released = before - after
print(f"GPU memory  before close: {before:.2f} GB  after close+del: {after:.2f} GB")
print(f"GPU memory released:      {released:.2f} GB")
print()
if released > 0.05:
    print(f"✅ close() released ~{released:.2f} GB (hooks + capture buffers + distributed state).")
else:
    print("⚠ close() released < 50 MB — likely vLLM's model weights and KV cache pool")
    print("  staying resident. This is expected on vLLM 0.20.2; restart the runtime")
    print("  for a clean GPU.")

## Step 10 — Batched capture (`enable_batching=True`)

The default path above is compile-validated but single-prompt. The throughput
path for SAE/probe data collection is `enable_batching=True`: `enforce_eager`
(no torch.compile), per-request accumulation across chunked prefill, and
`batch_size > 1`. Boot a second bridge in eager mode and validate it.

> Boots a fresh vLLM engine. If the compiled bridge above didn't fully release
> (vLLM 0.20.x keeps weights resident — see Step 9), restart the runtime first
> and run Setup + this section alone.

In [ ]:
bridge_b = RemoteBridge.boot_vllm(
    MODEL,
    dtype=DTYPE,
    gpu_memory_utilization=0.5,
    max_model_len=2048,
    # Small cap so a long prompt below forces chunked prefill (multi-forward).
    max_num_batched_tokens=512,
    enable_batching=True,
)
print(f"Batched bridge booted (enforce_eager). Fireable hooks: {len(bridge_b._driver.supported_hook_points)}")
SAMPLE_HOOK = sorted(bridge_b._driver.supported_hook_points)[0]
print(f"Sampling correctness on hook: {SAMPLE_HOOK!r}")

### 10a — Batch parity

Each row's real-token activations must equal the same prompt run alone. vLLM
computes each request independently under PagedAttention, so right-padding the
batch is a pure cache-assembly artifact — real tokens are padding-independent.

In [ ]:
PROMPTS = ["The capital of France is", "The quick brown fox jumps over the", "Hello"]
batch_tokens = [bridge_b.to_tokens(p)[0].tolist() for p in PROMPTS]

_, cache_batch = bridge_b.run_with_cache(batch_tokens)
print(f"batched cache '{SAMPLE_HOOK}' shape: {tuple(cache_batch[SAMPLE_HOOK].shape)}  (batch, max_seq, d_model)")

TARGET_REL_L2 = 5e-3
for k, ids in enumerate(batch_tokens):
    L = len(ids)
    _, cache_single = bridge_b.run_with_cache([ids])
    bt = cache_batch[SAMPLE_HOOK][k, :L].float()
    st = cache_single[SAMPLE_HOOK][0, :L].float()
    rel = (bt - st).norm().item() / (st.norm().item() or 1.0)
    print(f"  row {k} ({L:>2} tok): rel L2 = {rel:.3e}")
    assert rel < TARGET_REL_L2, f"row {k} batched != single (rel {rel:.3e})"
    # Pad positions past the real length must be zero.
    assert torch.equal(cache_batch[SAMPLE_HOOK][k, L:], torch.zeros_like(cache_batch[SAMPLE_HOOK][k, L:]))
print("✅ batched rows match single-prompt runs; pad positions zeroed.")

### 10b — Chunked-prefill accumulation

A prompt longer than `max_num_batched_tokens` splits across forwards. The eager
hook accumulates per-request slices and `torch.cat`s them — a single-buffer
overwrite (the compiled design) would keep only the last chunk. Assert the
captured sequence length equals the full prompt length.

In [ ]:
base_ids = bridge_b.to_tokens("The quick brown fox jumps over the lazy dog. ")[0].tolist()
long_ids = (base_ids * 60)[:600]  # > max_num_batched_tokens=512 → forces chunking
print(f"prompt length {len(long_ids)} tok  >  max_num_batched_tokens=512")

_, cache_long = bridge_b.run_with_cache([long_ids])
seq = cache_long[SAMPLE_HOOK].shape[1]
print(f"captured seq length: {seq}")
assert seq == len(long_ids), f"chunked prefill lost tokens: captured {seq} of {len(long_ids)} — accumulation broken"
assert torch.isfinite(cache_long[SAMPLE_HOOK]).all(), "non-finite values in accumulated capture"
print("✅ chunked-prefill accumulation reconstructs the full sequence.")

### 10c — Batched interventions (global)

Interventions in batched mode are global across the batch. A `suppress` on the
embedding shifts every row's prediction; a clean batch reverts.

In [ ]:
logits_clean, _ = bridge_b.run_with_cache(batch_tokens)
clean_argmax = [int(logits_clean[k, len(ids) - 1].argmax().item()) for k, ids in enumerate(batch_tokens)]

logits_supp, cache_supp = bridge_b.run_with_cache(
    batch_tokens, intervene={"embed.hook_out": {"op": "suppress"}}
)
supp_argmax = [int(logits_supp[k, len(ids) - 1].argmax().item()) for k, ids in enumerate(batch_tokens)]
embed_max = cache_supp["embed.hook_out"].abs().max().item()
print(f"clean argmax:    {clean_argmax}")
print(f"suppress argmax: {supp_argmax}   embed |max|={embed_max:.6f} (→0)")
assert embed_max == 0.0, "suppress did not zero embed across the batch"
assert supp_argmax != clean_argmax, "suppress did not shift any row's prediction"

logits_revert, _ = bridge_b.run_with_cache(batch_tokens)
revert_argmax = [int(logits_revert[k, len(ids) - 1].argmax().item()) for k, ids in enumerate(batch_tokens)]
print(f"revert argmax:   {revert_argmax}   matches clean: {revert_argmax == clean_argmax}")
assert revert_argmax == clean_argmax, "intervention persisted across batched calls"
print("✅ batched interventions apply globally and reset cleanly.")

In [ ]:
bridge_b.close()
del bridge_b, cache_batch, cache_long, logits_clean, logits_supp, cache_supp, logits_revert
gc.collect(); torch.cuda.empty_cache()
print(f"GPU memory after batched-bridge close: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Summary

If all asserts above passed, the v4 Driver-protocol vLLM backend is sound on this architecture:

- `boot_vllm` returns `RemoteBridge` end-to-end.
- `collective_rpc → tl_read_captures` populates the cache.
- Greedy argmax matches HF (greedy parity).
- Per-hook L2 < 5e-3 vs HF across every fireable hook (hook-fidelity gate; `ln_final.hook_normalized` exempted — see overlay docstring).
- Affine interventions (suppress / scale / add / set) apply per-forward and reset cleanly.
- Each hook fires exactly once per forward under compile (no double-fire / silent overwrite).
- Back-to-back captures are deterministic (stream-safe).
- GPU lifetime behaves as expected for vLLM 0.20.x (full release needs a runtime restart).
- **Batched mode** (`enable_batching=True`, Step 10): `batch_size > 1` parity vs single-prompt runs, chunked-prefill accumulation reconstructs full sequences, and global interventions apply + reset across the batch.

Known divergences (documented, not bugs): `ln_final.hook_normalized` post-weight vs pre-weight, Gemma `embed.hook_out` scaling. Both reconciled by conversions noted in `sources.vllm.overlays.decoder_only`.

Out of scope: multi-token generation (`max_new_tokens > 1`), per-request (non-global) batched interventions, tensor/pipeline parallelism.